# STARE — Multi-Seed Robustness Analysis

This notebook shows how to:
1. Load multi-seed experiment results
2. Visualize cross-seed performance distributions
3. Perform pairwise statistical tests across scenarios
4. Generate publication-ready LaTeX tables

**Prerequisites:** Install the project first with `pip install -e .` from the repo root.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from evaluation.analyze_multiseed import (
    load_all_runs, aggregate, pairwise_tests,
    plot_boxplots, plot_bar_with_ci,
    latex_main_table, latex_ci_table,
)

## 1. Load Multi-Seed Results

Point to the directory containing per-scenario CSVs from `python -m evaluation.multiseed`.
Each subdirectory (e.g. `turb_ppo/`) should have an `all_runs.csv`.

In [ ]:
# Path to multi-seed results
RESULTS_DIR = '../results/multi_seed'

df = load_all_runs(RESULTS_DIR)
print(f"Loaded {len(df)} runs: {df['agent'].nunique()} agents × {df['scenario'].nunique()} scenarios")
print(f"Seeds per cell: {df.groupby(['agent','scenario']).size().unique()}")
df.head()

## 2. Aggregated Summary (Mean ± Std)

In [ ]:
agg = aggregate(df)

display_cols = ['agent', 'scenario', 'n_seeds',
                'SR_mean', 'SR_std', 'Sortino_mean', 'AR_mean',
                'MaxDrawdown_mean', 'PSR_mean', 'DSR_mean']
agg[display_cols].style.format({
    'SR_mean': '{:.3f}', 'SR_std': '{:.3f}',
    'Sortino_mean': '{:.3f}', 'AR_mean': '{:.1%}',
    'MaxDrawdown_mean': '{:.1%}', 'PSR_mean': '{:.3f}', 'DSR_mean': '{:.3f}'
})

## 3. Boxplots — Cross-Seed Distributions

In [ ]:
from matplotlib.patches import Patch

SCENARIOS = ('turb', 'noturb', 'synth')
AGENTS = ('a2c', 'ppo', 'ddpg', 'td3')
colors = {'turb': '#4C72B0', 'noturb': '#DD8452', 'synth': '#55A868'}

for metric in ('SR', 'PSR', 'MaxDrawdown'):
    fig, ax = plt.subplots(figsize=(8, 4))
    offset = {'turb': -0.25, 'noturb': 0.0, 'synth': 0.25}
    for i, agent in enumerate(AGENTS):
        for sc in SCENARIOS:
            vals = df[(df['agent'] == agent) & (df['scenario'] == sc)][metric].to_numpy()
            if len(vals) == 0:
                continue
            ax.boxplot(vals, positions=[i + offset[sc]], widths=0.2,
                       patch_artist=True,
                       boxprops=dict(facecolor=colors[sc], alpha=0.6),
                       medianprops=dict(color='black'))
    ax.set_xticks(range(len(AGENTS)))
    ax.set_xticklabels([a.upper() for a in AGENTS])
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} — multi-seed distribution')
    ax.legend(handles=[Patch(facecolor=colors[s], alpha=0.6, label=s) for s in SCENARIOS])
    ax.grid(True, axis='y', alpha=0.3)
    fig.tight_layout()
    plt.show()

## 4. Pairwise Statistical Tests

In [ ]:
tests = pairwise_tests(df, metric='SR')
if not tests.empty:
    display(tests.style.format({
        'mean_diff': '{:.3f}', 'cohen_d': '{:.3f}',
        'welch_p': '{:.4f}', 'mw_p': '{:.4f}'
    }))
else:
    print('Not enough scenarios for pairwise tests.')

## 5. LaTeX Table (Copy-Paste to Paper)

In [ ]:
print(latex_main_table(agg))